# LeetCode 1860 — Incremental Memory Leak bằng Python

Link đề: https://leetcode.com/problems/incremental-memory-leak/ · Độ khó: **Medium** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def memLeak(self, memory1: int, memory2: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ một đề **mô phỏng**: thứ tự hành động mỗi giây, quy tắc hòa (thanh 1 được ưu tiên), điều kiện dừng.
2. Ước lượng số bước mô phỏng bằng **tổng cấp số cộng**: `1 + 2 + … + t ≈ t²/2 ≤ m₁ + m₂` ⇒ chỉ `O(√(m₁+m₂))` bước — đủ nhanh dù giá trị tới `2³¹`.
3. Nâng cấp lên lời giải **toán học** hai pha: pha 1 (chỉ một thanh bị cấp phát) và pha 2 (hai thanh **luân phiên**), giải bằng **binary search** hoặc **căn bậc hai nguyên**.
4. Chứng minh tính chất "luân phiên" — một ví dụ hay về **bất biến** trong mô phỏng.
5. Kiểm chứng mọi cách bằng mô phỏng trên dữ liệu ngẫu nhiên, kể cả các trường hợp hòa và bằng 0.

## 2. Đề bài & ví dụ

Có hai thanh nhớ với `memory1` và `memory2` bit trống. Một chương trình lỗi tiêu thụ bộ nhớ tăng dần: tại giây thứ `i` (bắt đầu từ 1), cấp phát `i` bit cho thanh có **nhiều bộ nhớ trống hơn** (nếu bằng nhau thì cấp cho **thanh 1**). Nếu không thanh nào còn ít nhất `i` bit, chương trình **crash**.

Trả về `[crashTime, memory1_crash, memory2_crash]`: thời điểm crash (giây) và bộ nhớ trống của hai thanh lúc đó.

**Ví dụ 1:** `memory1 = 2, memory2 = 2` → `[3,1,0]`.
- Giây 1: hòa ⇒ thanh 1, còn `(1, 2)`. Giây 2: thanh 2 nhiều hơn, còn `(1, 0)`. Giây 3: không thanh nào có 3 bit ⇒ crash.

**Ví dụ 2:** `memory1 = 8, memory2 = 11` → `[6,0,4]`.
- Giây 1, 2: thanh 2 (`11→10→8`). Giây 3: hòa `8 = 8` ⇒ thanh 1 (`8→5`). Giây 4: thanh 2 (`8→4`). Giây 5: thanh 1 (`5→0`). Giây 6: crash với `(0, 4)`.

**Ràng buộc:** `0 ≤ memory1, memory2 ≤ 2³¹ − 1`.

In [1]:
import math
import random
import time
from typing import List

examples = [((2, 2), [3, 1, 0]), ((8, 11), [6, 0, 4])]
for (m1, m2), exp in examples:
    print(f"memory1={m1}, memory2={m2} -> kỳ vọng {exp}")

memory1=2, memory2=2 -> kỳ vọng [3, 1, 0]
memory1=8, memory2=11 -> kỳ vọng [6, 0, 4]


## 3. Kiến thức nền

### 3.1 Tổng cấp số cộng

`1 + 2 + … + t = t(t+1)/2`. Sau `t` giây, tổng bộ nhớ đã cấp phát là `t(t+1)/2`, nên chương trình chắc chắn crash trước khi `t(t+1)/2 > m₁ + m₂`, tức `t ≲ √(2(m₁+m₂))`.

In [2]:
M = 2 * (2 ** 31 - 1)
t = math.isqrt(2 * M)
while t * (t + 1) // 2 > M:
    t -= 1
print(f"m1 + m2 = {M} -> tối đa khoảng {t} giây (≈ √(2M) = {math.sqrt(2 * M):.0f})")

m1 + m2 = 4294967294 -> tối đa khoảng 92681 giây (≈ √(2M) = 92682)


### 3.2 Căn bậc hai nguyên `math.isqrt`

`math.isqrt(x)` trả `⌊√x⌋` **chính xác** cho số nguyên lớn tùy ý (không sai số dấu phẩy động). Dùng để giải bất phương trình bậc hai `t² + bt ≤ c` trong số nguyên.

### 3.3 Binary search trên hàm đơn điệu

Tìm `t` lớn nhất thỏa `f(t) ≤ c` với `f` tăng: tìm kiếm nhị phân trên `[0, hi]`, `O(log hi)`.

In [3]:
def largest_t(f, c, hi=1 << 20):
    lo = 0                                  # f(0) <= c giả định đúng
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if f(mid) <= c:
            lo = mid
        else:
            hi = mid - 1
    return lo

print("t lớn nhất với t(t+1)/2 <= 10:", largest_t(lambda t: t * (t + 1) // 2, 10))

t lớn nhất với t(t+1)/2 <= 10: 4


### 3.4 Quy tắc hòa trong mô phỏng

"Nhiều hơn, nếu bằng thì thanh 1" ⇔ chọn thanh 1 khi `m1 >= m2`, ngược lại thanh 2. Một dấu `>` thay cho `>=` sẽ đổi kết quả ngay ở Ví dụ 1.

## 4. Ý tưởng nền tảng / trực giác

Chỉ cần làm đúng theo đề: mỗi giây chọn thanh lớn hơn (ưu tiên thanh 1), kiểm tra đủ `i` bit chưa, trừ đi, tăng `i`. Câu hỏi là **có đủ nhanh không**: số giây tối đa ≈ `√(2 · 4.3·10⁹) ≈ 92 682` — rất nhỏ. Vậy **mô phỏng trực tiếp** là lời giải chuẩn.

Tính tay **Ví dụ 2** `(8, 11)`:

| giây i | trước (m1, m2) | chọn | sau |
|---|---|---|---|
| 1 | (8, 11) | thanh 2 | (8, 10) |
| 2 | (8, 10) | thanh 2 | (8, 8) |
| 3 | (8, 8) | hòa ⇒ thanh 1 | (5, 8) |
| 4 | (5, 8) | thanh 2 | (5, 4) |
| 5 | (5, 4) | thanh 1 | (0, 4) |
| 6 | (0, 4) | 4 < 6 ⇒ **crash** | `[6, 0, 4]` |

Để ý: từ giây 3 trở đi hai thanh **luân phiên** 1, 2, 1… — đây là mầm của lời giải toán học.

In [4]:
m1, m2, i = 8, 11, 1
while True:
    pick = 1 if m1 >= m2 else 2
    if max(m1, m2) < i:
        print(f"i={i}: crash -> {[i, m1, m2]}"); break
    if pick == 1: m1 -= i
    else: m2 -= i
    print(f"i={i}: thanh {pick} -> ({m1}, {m2})"); i += 1

i=1: thanh 2 -> (8, 10)
i=2: thanh 2 -> (8, 8)
i=3: thanh 1 -> (5, 8)
i=4: thanh 2 -> (5, 4)
i=5: thanh 1 -> (0, 4)
i=6: crash -> [6, 0, 4]


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng từng câu chữ của đề

Viết sát đề nhất có thể: kiểm tra "không thanh nào có ít nhất `i` bit" bằng hai điều kiện riêng, rồi chọn thanh theo quy tắc.

In [5]:
def memleak_bruteforce(memory1, memory2):
    i = 1
    while True:
        if memory1 < i and memory2 < i:          # không thanh nào đủ i bit
            return [i, memory1, memory2]
        if memory1 >= memory2:                   # nhiều hơn, hòa thì thanh 1
            memory1 -= i
        else:
            memory2 -= i
        i += 1

for (m1, m2), exp in examples:
    got = memleak_bruteforce(m1, m2); print(got, "kỳ vọng", exp); assert got == exp

[3, 1, 0] kỳ vọng [3, 1, 0]
[6, 0, 4] kỳ vọng [6, 0, 4]


**Vì sao chọn thanh lớn hơn rồi trừ mà không cần kiểm tra thanh đó đủ bit?** Nếu không crash thì **có ít nhất một** thanh `≥ i`, và thanh **lớn hơn** chắc chắn là một thanh như vậy. Vì thế bộ nhớ không bao giờ âm.

**Độ phức tạp:** `O(√(m₁+m₂))` thời gian (≈ 9·10⁴ vòng tệ nhất), `O(1)` bộ nhớ. Đây đã là lời giải được chấp nhận.

## 6. Ý tưởng thuật toán / bước đột phá

Muốn nhanh hơn nữa (`O(1)` hoặc `O(log)`), quan sát cấu trúc của quá trình:

**Pha 1:** ban đầu thanh lớn hơn (gọi là `X`, thanh còn lại `Y`) nhận liên tiếp `1, 2, …, k` cho tới khi nó **không còn lớn hơn** `Y` (theo quy tắc hòa). Số `k` tìm được bằng giải `X − (j−1)j/2 ≥ Y` (nếu `X` là thanh 1) hoặc `> Y` (nếu `X` là thanh 2).

**Pha 2 (bất biến luân phiên):** từ giây `i = k + 1`, thanh `Y` được chọn, rồi `X`, rồi `Y`… **luân phiên** mãi đến khi crash. Lý do: hiệu giữa hai thanh luôn nhỏ hơn lượng cấp phát hiện tại, nên thanh vừa bị trừ luôn trở thành thanh **nhỏ hơn hẳn** (xem chứng minh mục 10).

Trong pha 2, `Y` nhận `i, i+2, i+4, …` và `X` nhận `i+1, i+3, …`. Số lần `Y` chịu được: `t` lớn nhất với `t·i + t(t−1) ≤ Y`; số lần `X` chịu được: `u` lớn nhất với `u·(i+1) + u(u−1) ≤ X'`. Crash ở giây `min(i + 2t, i + 1 + 2u)`.

**Invariant pha 2:** trước mỗi giây `s`, thanh sắp được chọn lớn hơn thanh kia **ít hơn `s`** (hiệu `< s`).

### 6.1 Nhìn hai pha trên vài cặp cụ thể

In chuỗi thanh được chọn mỗi giây (`1` hoặc `2`) cho vài cặp: luôn là một khối cùng chữ số ở đầu (pha 1), sau đó đan xen tuyệt đối (pha 2). Cặp lệch nhau nhiều ⇒ pha 1 dài; cặp bằng nhau ⇒ pha 1 chỉ 1 giây.

In [6]:
def pick_string(m1, m2):
    i, out = 1, []
    while max(m1, m2) >= i:
        if m1 >= m2: m1 -= i; out.append("1")
        else: m2 -= i; out.append("2")
        i += 1
    return "".join(out)

for a, b in [(8, 11), (50, 50), (200, 10), (10, 200), (0, 40)]:
    sstr = pick_string(a, b)
    k = len(sstr) - len(sstr.lstrip(sstr[0])) if sstr else 0
    print(f"({a:>3}, {b:>3}): {sstr:<30} pha 1 dài {k}")

(  8,  11): 22121                          pha 1 dài 2
( 50,  50): 1212121212121                  pha 1 dài 1
(200,  10): 1111111111111111111            pha 1 dài 19
( 10, 200): 2222222222222222222            pha 1 dài 19
(  0,  40): 22222222                       pha 1 dài 8


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng gọn (bản nộp)

Gộp điều kiện crash thành `max(m1, m2) < i` — tương đương "cả hai `< i`".

In [7]:
def memleak_sim(memory1, memory2):
    i = 1
    while max(memory1, memory2) >= i:
        if memory1 >= memory2:
            memory1 -= i
        else:
            memory2 -= i
        i += 1
    return [i, memory1, memory2]

print([memleak_sim(*a) for a, _ in examples])

[[3, 1, 0], [6, 0, 4]]


### 7.2 Cách 3 — Hai pha + binary search

Tổng quát hóa cho cả hai trường hợp `X` là thanh 1 hay thanh 2, kể cả crash ngay trong pha 1 (chỉ xảy ra khi `X` cạn sớm, ví dụ `(0, 0)`).

In [8]:
def memleak_two_phase(memory1, memory2, solve=None):
    solve = solve or (lambda f, c: largest_t(f, c, 1 << 18))
    x_is_1 = memory1 >= memory2
    X, Y = (memory1, memory2) if x_is_1 else (memory2, memory1)
    # Pha 1: X được chọn ở giây j khi X - (j-1)j/2 >= Y (thanh 1) hoặc > Y (thanh 2)
    slack = X - Y if x_is_1 else X - Y - 1               # điều kiện: (j-1)j/2 <= slack
    k = solve(lambda j: (j - 1) * j // 2, slack) if slack >= 0 else 0
    kc = solve(lambda j: j * (j + 1) // 2, X)            # X không âm sau j bước
    def pack(t, x, y):
        return [t, x, y] if x_is_1 else [t, y, x]
    if kc < k:                                           # crash ngay trong pha 1
        j = kc + 1
        return pack(j, X - kc * (kc + 1) // 2, Y)
    Xp, i = X - k * (k + 1) // 2, k + 1
    # Pha 2: Y nhận i, i+2, ...; X nhận i+1, i+3, ...
    t = solve(lambda q: q * i + q * (q - 1), Y)
    u = solve(lambda q: q * (i + 1) + q * (q - 1), Xp)
    if i + 2 * t <= i + 1 + 2 * u:                       # Y cạn trước
        crash, ty, tx = i + 2 * t, t, t
    else:
        crash, ty, tx = i + 1 + 2 * u, u + 1, u
    y_left = Y - (ty * i + ty * (ty - 1))
    x_left = Xp - (tx * (i + 1) + tx * (tx - 1))
    return pack(crash, x_left, y_left)

print([memleak_two_phase(*a) for a, _ in examples])

[[3, 1, 0], [6, 0, 4]]


Giải thích nhánh crash pha 2:
- Nếu `Y` cạn trước (giây `i + 2t`, lượt thứ `t+1` của `Y`): lúc đó `Y` đã nhận `t` lần, `X` đã nhận `t` lần (các giây `i+1, …, i+2t−1`).
- Nếu `X` cạn trước (giây `i + 1 + 2u`): `X` đã nhận `u` lần, `Y` đã nhận `u + 1` lần.

Khi crash, thanh được chọn là thanh **lớn hơn**; nó không đủ thì thanh kia càng không đủ ⇒ đúng là "cả hai đều thiếu".

**Độ phức tạp Cách 3:** vài lần binary search ⇒ `O(log M)`.

### 7.3 Cách 4 — Hai pha + căn bậc hai nguyên (O(1))

Thay binary search bằng công thức nghiệm của `q² + bq ≤ c` (với `b = i − 1` hoặc `i`), rồi chỉnh ±1 cho chắc chắn.

In [9]:
def isqrt_solve(f, c):
    # f là đa thức bậc hai tăng với f(0) = 0; ước lượng q ≈ √(2c) rồi chỉnh.
    if c < 0:
        return 0
    q = math.isqrt(2 * c) + 2
    while q > 0 and f(q) > c:
        q -= 1
    while f(q + 1) <= c:
        q += 1
    return q

def memleak_isqrt(memory1, memory2):
    return memleak_two_phase(memory1, memory2, solve=isqrt_solve)

print([memleak_isqrt(*a) for a, _ in examples])

[[3, 1, 0], [6, 0, 4]]


Phiên bản "ước lượng rồi chỉnh" ở trên đúng nhưng **chưa phải `O(1)`**: `√(2c)` chỉ là cận trên thô, với `f(q) ≈ q²` vòng `while` phải lùi khoảng `0.41·√c` bước. Để thật sự `O(1)`, giải đúng phương trình bậc hai `q² + bq ≤ c`: `q = ⌊(−b + √(b² + 4c)) / 2⌋` tính bằng `isqrt`, rồi chỉnh ±1 phòng sai lệch do làm tròn:

In [10]:
def quad_solve_factory(b):
    # q lớn nhất với q² + b q <= c
    def solve(c):
        if c < 0:
            return 0
        q = (math.isqrt(b * b + 4 * c) - b) // 2
        while q * q + b * q > c: q -= 1
        while (q + 1) ** 2 + b * (q + 1) <= c: q += 1
        return q
    return solve

def memleak_closed(memory1, memory2):
    x_is_1 = memory1 >= memory2
    X, Y = (memory1, memory2) if x_is_1 else (memory2, memory1)
    slack = X - Y if x_is_1 else X - Y - 1
    # (j-1)j/2 <= slack  <=>  j² - j <= 2 slack
    k = quad_solve_factory(-1)(2 * slack) if slack >= 0 else 0
    kc = quad_solve_factory(1)(2 * X)                     # j² + j <= 2X
    pack = (lambda t, x, y: [t, x, y]) if x_is_1 else (lambda t, x, y: [t, y, x])
    if kc < k:
        return pack(kc + 1, X - kc * (kc + 1) // 2, Y)
    Xp, i = X - k * (k + 1) // 2, k + 1
    t = quad_solve_factory(i - 1)(Y)                      # q² + (i-1) q <= Y
    u = quad_solve_factory(i)(Xp)                         # q² + i q <= X'
    if 2 * t <= 1 + 2 * u:
        crash, ty, tx = i + 2 * t, t, t
    else:
        crash, ty, tx = i + 1 + 2 * u, u + 1, u
    return pack(crash, Xp - (tx * tx + i * tx), Y - (ty * ty + (i - 1) * ty))

print([memleak_closed(*a) for a, _ in examples])

[[3, 1, 0], [6, 0, 4]]


Kiểm tra lại biến đổi: `q·i + q(q−1) = q² + (i−1)q` và `q(i+1) + q(q−1) = q² + i·q`. Mỗi hàm `solve` chỉ chỉnh tối đa vài bước nên **`O(1)`** phép tính trên số nguyên (cộng chi phí `isqrt` cho số `≤ 2⁶⁴`).

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 2** `(8, 11)`: in các đại lượng trung gian.

In [11]:
def trace_two_phase(m1, m2):
    x_is_1 = m1 >= m2
    X, Y = (m1, m2) if x_is_1 else (m2, m1)
    slack = X - Y if x_is_1 else X - Y - 1
    k = largest_t(lambda j: (j - 1) * j // 2, slack) if slack >= 0 else 0
    print(f"X = thanh {1 if x_is_1 else 2} ({X}), Y = {Y}, slack = {slack} -> pha 1 kéo dài k = {k} giây")
    Xp, i = X - k * (k + 1) // 2, k + 1
    print(f"Sau pha 1: X' = {Xp}, bắt đầu luân phiên từ giây i = {i} với Y được chọn trước")
    t = largest_t(lambda q: q * i + q * (q - 1), Y)
    u = largest_t(lambda q: q * (i + 1) + q * (q - 1), Xp)
    print(f"Y chịu được t = {t} lượt (giây {[i + 2 * s for s in range(t)]}), X chịu được u = {u} lượt (giây {[i + 1 + 2 * s for s in range(u)]})")
    print(f"crash = min(i+2t, i+1+2u) = min({i + 2 * t}, {i + 1 + 2 * u})")
    return memleak_two_phase(m1, m2)

r = trace_two_phase(8, 11); print("kết quả:", r); assert r == [6, 0, 4]

X = thanh 2 (11), Y = 8, slack = 2 -> pha 1 kéo dài k = 2 giây
Sau pha 1: X' = 8, bắt đầu luân phiên từ giây i = 3 với Y được chọn trước
Y chịu được t = 2 lượt (giây [3, 5]), X chịu được u = 1 lượt (giây [4])
crash = min(i+2t, i+1+2u) = min(7, 6)
kết quả: [6, 0, 4]


| đại lượng | giá trị | ý nghĩa |
|---|---|---|
| X, Y | thanh 2 = 11, thanh 1 = 8 | thanh 2 lớn hơn ban đầu |
| slack | 11 − 8 − 1 = 2 | thanh 2 cần **lớn hơn hẳn** |
| k | 2 | giây 1, 2 cho thanh 2 (`(j−1)j/2 ≤ 2` với `j ≤ 2`) |
| X' , i | 8, 3 | từ giây 3 luân phiên, thanh 1 (Y) trước |
| t | 2 | Y nhận giây 3, 5 (`3 + 5 = 8 ≤ 8`) |
| u | 1 | X nhận giây 4 (`4 ≤ 8`, nhưng `4 + 6 > 8`) |
| crash | min(7, 6) = 6 | X cạn ở lượt thứ 2 (giây 6) |

Bộ nhớ còn: `Y = 8 − 8 = 0`, `X = 8 − 4 = 4` ⇒ `[6, 0, 4]` ✓.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — mô phỏng, `O(√(m₁+m₂))`, ngắn và không thể sai công thức.

In [12]:
class Solution:
    def memLeak(self, memory1: int, memory2: int) -> List[int]:
        i = 1
        while max(memory1, memory2) >= i:
            if memory1 >= memory2:
                memory1 -= i
            else:
                memory2 -= i
            i += 1
        return [i, memory1, memory2]

for (m1, m2), exp in examples:
    got = Solution().memLeak(m1, m2); print(got, exp); assert got == exp

[3, 1, 0] [3, 1, 0]
[6, 0, 4] [6, 0, 4]


## 10. Vì sao thuật toán đúng

**Cách 1/2.** Là mô phỏng trực tiếp: mỗi vòng thực hiện đúng một giây theo đề. Điều kiện `max(m1, m2) ≥ i` ⇔ "có ít nhất một thanh đủ `i` bit" ⇔ không crash. Thanh được chọn là thanh lớn hơn (hòa ⇒ thanh 1) nên luôn đủ bit. Vòng lặp kết thúc vì tổng bộ nhớ giảm `i ≥ 1` mỗi giây.

**Bất biến luân phiên (Cách 3/4).** Gọi `D_s` = (thanh được chọn ở giây `s`) − (thanh kia), trước khi trừ. Ta chứng minh trong pha 2: `0 ≤ D_s < s`, và quy tắc hòa không còn ảnh hưởng.
- *Bắt đầu pha 2 (giây `i = k+1`):* pha 1 dừng vì `X'` không còn được chọn ⇒ `Y ≥ X'` (đúng thứ tự hòa). Ở giây `k`, `X` còn được chọn: `X' + k ≥ Y` ⇒ `D_i = Y − X' ≤ k < i`.
- *Bước quy nạp:* thanh được chọn sau khi trừ còn `A − s`, thanh kia `B` với `0 ≤ A − B < s` ⇒ `B − (A − s) = s − (A − B) > 0`: thanh kia lớn hơn **hẳn** ⇒ nó được chọn ở giây `s+1` (không phụ thuộc quy tắc hòa), với hiệu `D_{s+1} = s − D_s ≤ s < s + 1`. ✓

Vậy từ giây `i` hai thanh luân phiên, `Y` trước. Crash ở lượt đầu tiên thanh đến lượt không đủ bit — là giây nhỏ nhất trong hai "giây cạn" của `Y` và `X`. ∎

### 10.1 Kiểm chứng thực nghiệm bất biến luân phiên

Ghi lại thanh được chọn mỗi giây bằng mô phỏng và kiểm tra: sau một đoạn đầu chọn cùng một thanh, dãy luôn đan xen `1, 2, 1, 2, …`, và hiệu trước mỗi giây luôn `< s` trong pha 2.

In [13]:
def pick_log(m1, m2):
    i, log, diffs = 1, [], []
    while max(m1, m2) >= i:
        p = 1 if m1 >= m2 else 2
        log.append(p); diffs.append(abs(m1 - m2))
        if p == 1: m1 -= i
        else: m2 -= i
        i += 1
    return log, diffs

rng = random.Random(1010)
for _ in range(3000):
    a, b = rng.randint(0, 3000), rng.randint(0, 3000)
    log, diffs = pick_log(a, b)
    k = 0
    while k < len(log) and log[k] == log[0]:
        k += 1
    assert all(log[s] != log[s - 1] for s in range(k, len(log)))
    assert all(diffs[s] < s + 1 for s in range(k, len(log)))
print("Pha 1 = một đoạn cùng thanh, pha 2 luân phiên với hiệu < s: đúng trên 3000 cặp")

Pha 1 = một đoạn cùng thanh, pha 2 luân phiên với hiệu < s: đúng trên 3000 cặp


## 11. Phân tích Time / Space Complexity

Gọi `M = memory1 + memory2 ≤ 2³² − 2`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng sát đề | `O(√M)` ≈ 9·10⁴ vòng | `O(1)` |
| 2. Mô phỏng gọn | `O(√M)` | `O(1)` |
| 3. Hai pha + binary search | `O(log M)` | `O(1)` |
| 4. Hai pha + `isqrt` | `O(1)` phép số học | `O(1)` |

In [14]:
BIG = 2 ** 31 - 1
cases = [(BIG, BIG), (BIG, 0), (0, BIG), (BIG, BIG - 1)]
for name, f in [("sim", Solution().memLeak), ("two_phase", memleak_two_phase), ("closed", memleak_closed)]:
    t = time.perf_counter(); res = [f(a, b) for a, b in cases]; dt = (time.perf_counter() - t) * 1000
    print(f"{name:<10} {res[0]} ... ({dt:.2f} ms cho {len(cases)} ca lớn)")

sim        [92681, 88047, 41707] ... (17.27 ms cho 4 ca lớn)
two_phase  [92681, 88047, 41707] ... (0.05 ms cho 4 ca lớn)
closed     [92681, 88047, 41707] ... (0.02 ms cho 4 ca lớn)


## 12. Bộ test

In [15]:
ALL = {"brute": memleak_bruteforce, "sim": memleak_sim, "two_phase": memleak_two_phase,
       "isqrt": memleak_isqrt, "closed": memleak_closed, "submit": Solution().memLeak}
def run_all(m1, m2, brute=True):
    res = {k: f(m1, m2) for k, f in ALL.items() if brute or k not in ("brute",)}
    vals = {tuple(v) for v in res.values()}
    assert len(vals) == 1, (m1, m2, res)
    return res["submit"]

for (m1, m2), exp in examples:
    assert run_all(m1, m2) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kỳ vọng đã kiểm bằng mô phỏng sát đề
edge = [((0, 0), [1, 0, 0]), ((1, 0), [2, 0, 0]), ((0, 1), [2, 0, 0]), ((5, 5), [4, 1, 3]),
        ((1, 1), [2, 0, 1]), ((3, 0), [3, 0, 0]), ((0, 3), [3, 0, 0]), ((4, 0), [3, 1, 0])]
for (m1, m2), exp in edge:
    got = run_all(m1, m2); assert got == exp == memleak_bruteforce(m1, m2), (m1, m2, got)
for a in range(60):
    for b in range(60):
        run_all(a, b)
rng = random.Random(1860)
for _ in range(1500):
    run_all(rng.randint(0, 10 ** 6), rng.randint(0, 10 ** 6))
for _ in range(30):
    run_all(rng.randint(0, 2 ** 31 - 1), rng.randint(0, 2 ** 31 - 1))
for a, b in cases:
    run_all(a, b)
print("Ví dụ, edge case, lưới 60x60, 1500 cặp ≤ 1e6, 30 cặp lớn, 4 cực trị: OK")

Ví dụ, edge case, lưới 60x60, 1500 cặp ≤ 1e6, 30 cặp lớn, 4 cực trị: OK


### 12.1 Kiểm tra tính chất

- Bảo toàn: `m1 + m2 − (m1' + m2') = (crash − 1)·crash / 2` (tổng đã cấp phát).
- Lúc crash cả hai `< crash`, và trước đó (giây `crash − 1`) có thanh đủ bit.
- Hai thanh không bao giờ âm.

In [16]:
rng = random.Random(121)
f = Solution().memLeak
for _ in range(2000):
    a, b = rng.randint(0, 10 ** 5), rng.randint(0, 10 ** 5)
    c, x, y = f(a, b)
    assert a + b - (x + y) == (c - 1) * c // 2
    assert x < c and y < c and x >= 0 and y >= 0
print("3 tính chất đúng trên 2000 cặp")

3 tính chất đúng trên 2000 cặp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sai quy tắc hòa (ưu tiên thanh 2)

In [17]:
def wrong_tie(m1, m2):
    i = 1
    while max(m1, m2) >= i:
        if m1 > m2: m1 -= i                     # SAI: hòa thì phải chọn thanh 1
        else: m2 -= i
        i += 1
    return [i, m1, m2]

for (m1, m2), exp in examples:
    print("sai :", wrong_tie(m1, m2), "| đúng:", exp)

sai : [3, 0, 1] | đúng: [3, 1, 0]
sai : [6, 4, 0] | đúng: [6, 0, 4]


**Vì sao sai:** đề ghi "nếu bằng nhau thì cấp từ thanh 1". **Sửa:** `if m1 >= m2`.

### Lỗi 2 — Trả về `i − 1` (giây cuối cùng thành công) thay vì giây crash

In [18]:
def wrong_time(m1, m2):
    i = 1
    while max(m1, m2) >= i:
        if m1 >= m2: m1 -= i
        else: m2 -= i
        i += 1
    return [i - 1, m1, m2]                       # SAI: crash ở giây i

for (m1, m2), exp in examples:
    print("sai :", wrong_time(m1, m2), "| đúng:", exp)

sai : [2, 1, 0] | đúng: [3, 1, 0]
sai : [5, 0, 4] | đúng: [6, 0, 4]


**Vì sao sai:** `crashTime` là giây **xảy ra crash**, tức giây đầu tiên không cấp phát được. **Sửa:** trả `i`.

### Lỗi 3 — Chọn thanh theo giá trị **ban đầu** thay vì hiện tại

In [19]:
def wrong_initial_choice(m1, m2):
    first = 1 if m1 >= m2 else 2                  # SAI: quyết định một lần
    i = 1
    while True:
        if first == 1 and m1 >= i: m1 -= i
        elif first == 2 and m2 >= i: m2 -= i
        elif m1 >= i: m1 -= i
        elif m2 >= i: m2 -= i
        else: return [i, m1, m2]
        i += 1

for (m1, m2), exp in examples:
    print("sai :", wrong_initial_choice(m1, m2), "| đúng:", exp)

sai : [3, 1, 0] | đúng: [3, 1, 0]
sai : [6, 3, 1] | đúng: [6, 0, 4]


**Vì sao sai:** đề so sánh bộ nhớ **hiện tại** mỗi giây. Ở Ví dụ 2 thanh 2 lớn hơn ban đầu nhưng từ giây 3 thanh 1 lại được chọn. **Sửa:** so sánh lại mỗi vòng.

### Lỗi 4 — Cấp cho thanh "đầu tiên còn đủ bit" (thay vì thanh lớn hơn)

In [20]:
def wrong_first_fit(m1, m2):
    i = 1
    while True:
        if m1 >= i: m1 -= i                       # SAI: không so sánh hai thanh
        elif m2 >= i: m2 -= i
        else: return [i, m1, m2]
        i += 1

for (m1, m2), exp in examples:
    print("sai :", wrong_first_fit(m1, m2), "| đúng:", exp)

sai : [3, 1, 0] | đúng: [3, 1, 0]
sai : [6, 2, 2] | đúng: [6, 0, 4]


**Vì sao sai:** đây là chiến lược "first fit", không phải "thanh nhiều bộ nhớ hơn". **Sửa:** so sánh `m1 >= m2`.

### Lỗi 5 — Công thức hai pha quên quy tắc hòa ở pha 1 (dùng `≥` cho cả thanh 2)

In [21]:
def wrong_phase1(memory1, memory2):
    x_is_1 = memory1 >= memory2
    X, Y = (memory1, memory2) if x_is_1 else (memory2, memory1)
    slack = X - Y                                          # SAI khi X là thanh 2: cần X - Y - 1
    k = largest_t(lambda j: (j - 1) * j // 2, slack, 1 << 18)
    kc = largest_t(lambda j: j * (j + 1) // 2, X, 1 << 18)
    pack = (lambda t, x, y: [t, x, y]) if x_is_1 else (lambda t, x, y: [t, y, x])
    if kc < k:
        return pack(kc + 1, X - kc * (kc + 1) // 2, Y)
    Xp, i = X - k * (k + 1) // 2, k + 1
    t = largest_t(lambda q: q * i + q * (q - 1), Y, 1 << 18)
    u = largest_t(lambda q: q * (i + 1) + q * (q - 1), Xp, 1 << 18)
    if 2 * t <= 1 + 2 * u:
        crash, ty, tx = i + 2 * t, t, t
    else:
        crash, ty, tx = i + 1 + 2 * u, u + 1, u
    return pack(crash, Xp - (tx * (i + 1) + tx * (tx - 1)), Y - (ty * i + ty * (ty - 1)))

m1, m2 = examples[1][0]
print("sai :", wrong_phase1(m1, m2), "| đúng:", examples[1][1])

sai : [6, 4, 0] | đúng: [6, 0, 4]


**Vì sao sai:** ở Ví dụ 2, sau giây 2 hai thanh **bằng nhau** `(8, 8)` — quy tắc hòa trao giây 3 cho thanh 1, tức pha 1 của thanh 2 phải dừng. Dùng `≥` khiến pha 1 kéo dài thêm một giây và phá vỡ bất biến. **Sửa:** `slack = X − Y − 1` khi `X` là thanh 2. Bài học: lời giải công thức dễ sai ở biên; luôn **đối chiếu mô phỏng**.

### Lỗi 6 — Lo ngại thừa: "giá trị tới 2³¹ nên mô phỏng sẽ TLE"

In [22]:
t = time.perf_counter(); r = Solution().memLeak(2 ** 31 - 1, 2 ** 31 - 1)
print(r, f"— mô phỏng ca lớn nhất mất {(time.perf_counter() - t) * 1000:.1f} ms")

[92681, 88047, 41707] — mô phỏng ca lớn nhất mất 5.4 ms


**Vì sao là ngộ nhận:** số **giây** chỉ tăng theo căn bậc hai của tổng bộ nhớ, không phải tuyến tính. Luôn ước lượng số vòng thật trước khi bỏ một lời giải đơn giản.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "mỗi bước tiêu tốn `i` tăng dần" ⇒ số bước `O(√tổng)` ⇒ mô phỏng thường đủ.
- Đọc kỹ quy tắc hòa và định nghĩa "thời điểm crash" (giây thất bại, không phải giây thành công cuối).
- Muốn `O(1)`: tách quá trình thành các **pha có cấu trúc** (ở đây: một thanh liên tiếp, rồi luân phiên), mỗi pha là tổng cấp số cộng ⇒ giải bất phương trình bậc hai bằng `isqrt`.
- Lời giải công thức: luôn viết kèm mô phỏng để đối chiếu ngẫu nhiên.

## 15. Biến thể của bài

1. **Ba (hoặc `k`) thanh nhớ**: mỗi giây chọn thanh lớn nhất (hòa ⇒ chỉ số nhỏ nhất) — mô phỏng với heap `O(√M · log k)`.
2. **Lượng cấp phát khác**: `i²` hay `2^i` — số bước càng ít, mô phỏng càng nhanh.
3. **Hỏi trạng thái ở giây `t` bất kỳ** (nhiều truy vấn) — dùng công thức hai pha (Bài tập 5).
4. **Các bài mô phỏng có cấu trúc lặp** như 2582 Pass the Pillow (chu kỳ) — tìm quy luật để nhảy cóc thay vì chạy từng bước.

### 15.1 Code biến thể: `k` thanh nhớ với heap

In [23]:
import heapq
def memleak_k(mems):
    heap = [(-m, idx) for idx, m in enumerate(mems)]       # lớn nhất trước, hòa -> chỉ số nhỏ
    heapq.heapify(heap)
    i = 1
    while -heap[0][0] >= i:
        m, idx = heapq.heappop(heap)
        heapq.heappush(heap, (m + i, idx))
        i += 1
    res = [0] * len(mems)
    for m, idx in heap:
        res[idx] = -m
    return [i] + res

def memleak_k_brute(mems):
    mems = list(mems); i = 1
    while max(mems) >= i:
        j = mems.index(max(mems)); mems[j] -= i; i += 1
    return [i] + mems

rng = random.Random(15)
for _ in range(500):
    ms = [rng.randint(0, 300) for _ in range(rng.randint(1, 4))]
    assert memleak_k(ms) == memleak_k_brute(ms)
    if len(ms) == 2:
        assert memleak_k(ms) == Solution().memLeak(*ms)
print("k thanh (heap) khớp vét cạn trên 500 test; k = 2 trùng bài gốc;", memleak_k([8, 11, 5]))

k thanh (heap) khớp vét cạn trên 500 test; k = 2 trùng bài gốc; [6, 0, 4, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng sát đề | Từng giây | `O(√M)` | `O(1)` | Kiểm chứng |
| 2. Mô phỏng gọn | `max(m1,m2) ≥ i` | `O(√M)` | `O(1)` | **Bản nộp** |
| 3. Hai pha + BS | Pha liên tiếp + luân phiên | `O(log M)` | `O(1)` | Nhiều truy vấn |
| 4. Hai pha + `isqrt` | Nghiệm bậc hai | `O(1)` | `O(1)` | Khi `M` rất lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `(0, 0)`; (b) `(1, 1)`; (c) `(3, 0)`; (d) `(5, 5)`.

**Bài 2 (tính tay):** Lập bảng từng giây cho `(10, 3)` và xác định `k` (độ dài pha 1).

**Bài 3 (điền TODO):** Hoàn thiện `step(m1, m2, i)` trả `(m1', m2')` sau giây `i`, hoặc `None` nếu crash.

**Bài 4 (tự viết lại):** Viết `alloc_log(m1, m2)` trả danh sách `(giây, thanh)` của mọi lần cấp phát thành công.

**Bài 5 (mini challenge):** Viết `state_at(m1, m2, s)` trả `(m1, m2)` **trước** giây `s` (giả sử chưa crash trước `s`) trong `O(log M)` bằng công thức hai pha, đối chiếu mô phỏng.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def step_todo(m1, m2, i):
    # TODO: nếu max(m1, m2) < i: trả None; nếu m1 >= m2: trừ m1; ngược lại trừ m2
    pass

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def alloc_log(m1, m2):
    # TODO: mô phỏng, mỗi giây ghi (i, 1 hoặc 2)
    pass

def state_at(m1, m2, s):
    # TODO: tính k (pha 1); nếu s - 1 <= k: X mất (s-1)s/2;
    #       ngược lại đếm số giây của Y (i, i+2, ...) và của X (i+1, ...) trong [i, s-1]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: giây 1 cần 1 bit. Mức 2: hòa ⇒ thanh 1. Mức 3: (a) `[1,0,0]`; (b) `[2,0,1]`; (c) `[3,0,0]` (`3 → 2 → 0`); (d) `[4,1,3]`.

**Bài 2.** Mức 1: thanh 1 nhận liên tiếp khi còn `≥` thanh 2. Mức 2: `10 → 9 → 7 → 4`: giây 3 vẫn thanh 1 (`7 ≥ 3`), giây 4: `(4, 3)` ⇒ thanh 1 `→ 0`. Mức 3: `k = 4`; giây 5: `(0, 3)`, `3 < 5` ⇒ crash `[5, 0, 3]`.

**Bài 3.** Mức 1: điều kiện crash trước. Mức 2: quy tắc hòa. Mức 3: trả tuple mới, không sửa tại chỗ.

**Bài 4.** Mức 1: dựa trên Cách 2. Mức 2: thêm `log.append((i, p))`. Mức 3: kiểm tra độ dài log = `crash − 1`.

**Bài 5.** Mức 1: `k`, `X'`, `i` như Cách 3. Mức 2: trong `[i, s−1]` có `⌈(s−i)/2⌉` giây của `Y` và `⌊(s−i)/2⌋` giây của `X`. Mức 3: `Y` mất `q·i + q(q−1)`, `X` mất `q(i+1) + q(q−1)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
f = Solution().memLeak
b1 = [f(*a) for a in [(0, 0), (1, 1), (3, 0), (5, 5)]]
print("Bài 1:", b1); assert b1 == [[1, 0, 0], [2, 0, 1], [3, 0, 0], [4, 1, 3]]
log, _ = pick_log(10, 3)
print("Bài 2: thanh mỗi giây", log, "->", f(10, 3)); assert f(10, 3) == [5, 0, 3] and log == [1, 1, 1, 1]
def step_todo(m1, m2, i):
    if max(m1, m2) < i:
        return None
    return (m1 - i, m2) if m1 >= m2 else (m1, m2 - i)
st, i = (8, 11), 1
while (nxt := step_todo(*st, i)) is not None:
    st, i = nxt, i + 1
assert [i, *st] == [6, 0, 4]
print("Bài 3: OK")

Bài 1: [[1, 0, 0], [2, 0, 1], [3, 0, 0], [4, 1, 3]]
Bài 2: thanh mỗi giây [1, 1, 1, 1] -> [5, 0, 3]
Bài 3: OK


In [27]:
# Bài 4, 5
def alloc_log(m1, m2):
    i, log = 1, []
    while max(m1, m2) >= i:
        p = 1 if m1 >= m2 else 2
        if p == 1: m1 -= i
        else: m2 -= i
        log.append((i, p)); i += 1
    return log

def state_at(m1, m2, s):
    x_is_1 = m1 >= m2
    X, Y = (m1, m2) if x_is_1 else (m2, m1)
    slack = X - Y if x_is_1 else X - Y - 1
    k = largest_t(lambda j: (j - 1) * j // 2, slack, 1 << 18) if slack >= 0 else 0
    if s - 1 <= k:
        X -= (s - 1) * s // 2
    else:
        i = k + 1; X -= k * (k + 1) // 2
        qy, qx = (s - i + 1) // 2, (s - i) // 2
        Y -= qy * i + qy * (qy - 1); X -= qx * (i + 1) + qx * (qx - 1)
    return (X, Y) if x_is_1 else (Y, X)

rng = random.Random(5)
for _ in range(400):
    a, b = rng.randint(0, 500), rng.randint(0, 500)
    lg = alloc_log(a, b); c = Solution().memLeak(a, b)[0]
    assert len(lg) == c - 1
    x, y, i = a, b, 1
    for s in range(1, c + 1):
        assert state_at(a, b, s) == (x, y), (a, b, s)
        if s < c:
            if x >= y: x -= s
            else: y -= s
print("Bài 4, 5: OK |", alloc_log(8, 11), state_at(8, 11, 4))

Bài 4, 5: OK | [(1, 2), (2, 2), (3, 1), (4, 2), (5, 1)] (5, 8)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template mô phỏng:**

```python
i = 1
while max(m1, m2) >= i:          # còn thanh đủ i bit
    if m1 >= m2: m1 -= i         # hòa -> thanh 1
    else:        m2 -= i
    i += 1
return [i, m1, m2]               # i là giây crash
```

**Checklist:**
- [ ] Quy tắc hòa `m1 >= m2` (thanh 1 ưu tiên)?
- [ ] So sánh giá trị **hiện tại** mỗi giây?
- [ ] Trả giây **crash** (không phải giây thành công cuối)?
- [ ] Ước lượng số vòng `≈ √(2M)` — không ngại giá trị `2³¹`?
- [ ] Đã test `(0,0)`, một thanh bằng 0, hai thanh bằng nhau?